# Run Real Video VAE Latent Probe

本文件是 `real_video_vae_latent_probe` 运行入口。消费上游 processed dataset handoff 与 session-only VAE 模型, 并把 runner、checker、table rebuild 与结果打包全部委托给仓库模块。

## 00 Runtime Identity And User Config

集中声明本次 probe run 的运行模式、family 身份、processed dataset key、仓库根目录、session local 路径、模型来源与 formal 门禁开关。

`REPO_URL` 与 `REPO_ROOT` 控制 Colab 冷启动时的仓库 bootstrap；
`RUN_MODE` 与 `RUNTIME_PROFILE` 控制运行模式；
`WORKFLOW_KEY`、`STEP_KEY`、`FAMILY_ID` 用于结果包登记；
`PROCESSED_DATASET_KEY` 来自上游 processed dataset notebook 的最终 handoff；
`PROCESSED_DATASET_MANIFEST` 指向 session local 的 manifest；
`MODEL_ID` 指定 session-only 模型来源；
`REQUIRE_FORMAL_PASS=True` 表示正式检查失败时必须阻断打包。

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = os.environ.get('TSTW_REPO_URL', 'https://github.com/RICHAAARC/TSTW-VW.git')
REPO_BRANCH = os.environ.get('TSTW_REPO_BRANCH', 'main')
REPO_ROOT = Path(os.environ.get('TSTW_REPO_ROOT', '/content/TSTW-VW'))
REPO_PACKAGE_MARKER = REPO_ROOT / 'paper_workflow' / '__init__.py'
if not REPO_PACKAGE_MARKER.exists():
    if REPO_ROOT.exists():
        raise RuntimeError(f'Repository root exists but is incomplete: {REPO_ROOT}')
    clone_result = subprocess.run(
        [
            'git',
            'clone',
            '--depth',
            '1',
            '--branch',
            REPO_BRANCH,
            REPO_URL,
            str(REPO_ROOT),
        ],
        check=False,
        capture_output=True,
        text=True,
    )
    if clone_result.returncode != 0:
        raise RuntimeError(
            'repository bootstrap failed; set TSTW_REPO_URL to an authenticated clone URL or pre-clone the repository into TSTW_REPO_ROOT before running the notebook'
        )
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from paper_workflow.notebook_utils import real_video_vae_latent_probe_workflow as probe_workflow

RUN_MODE = 'formal'
RUNTIME_PROFILE = 'formal'
WORKFLOW_KEY = 'real_video_vae_latent_probe_completion_formal'
STEP_KEY = 'step02_run_real_video_vae_latent_probe'
FAMILY_ID = 'real_video_vae_latent_probe__formal__davis2017_trainval480p__utc_time__short_commit'
PROCESSED_DATASET_KEY = 'real_video_vae_latent_probe__davis2017_trainval480p__256x256__32f__8fps__freeze001'
DRIVE_ROOT = Path('/content/drive/MyDrive')
LOCAL_RUNTIME_ROOT = Path('/content/TSTW_runtime')
PROCESSED_DATASET_ROOT = DRIVE_ROOT / 'TSTW' / 'datasets' / 'processed' / PROCESSED_DATASET_KEY
LOCAL_DATASET_ROOT = LOCAL_RUNTIME_ROOT / 'datasets' / PROCESSED_DATASET_KEY
PROCESSED_DATASET_MANIFEST = LOCAL_DATASET_ROOT / 'dataset_manifest.json'
LOCAL_MODEL_ROOT = LOCAL_RUNTIME_ROOT / 'session_models' / 'autoencoder_kl'
FAMILY_ROOT = DRIVE_ROOT / 'TSTW' / 'results' / 'families' / FAMILY_ID
RUN_ROOT = LOCAL_RUNTIME_ROOT / 'runs' / 'real_video_vae_latent_probe_formal'
RUNTIME_CONFIG_PATH = RUN_ROOT / 'artifacts' / 'runtime_config.json'
SESSION_MODEL_MANIFEST_PATH = RUN_ROOT / 'artifacts' / 'session_model_manifest.json'
MODEL_ID = 'stabilityai/sd-vae-ft-mse'
REQUIRE_FORMAL_PASS = True

## 01 Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 02 Read Processed Dataset Registry

确认上游 processed dataset 根目录存在, 并打印当前 `PROCESSED_DATASET_KEY` 与 Drive 侧 processed dataset 路径。

In [ ]:
if not PROCESSED_DATASET_ROOT.exists():
    raise FileNotFoundError(PROCESSED_DATASET_ROOT)
print({
    'processed_dataset_root': str(PROCESSED_DATASET_ROOT),
    'processed_dataset_key': PROCESSED_DATASET_KEY,
})

## 03 Prepare Local Runtime Workspace

复制 processed dataset 到 local runtime, 创建 run root 与 family root, 并解析本次 runner 必须使用的 session local `dataset_manifest.json`。

In [ ]:
runtime_workspace_handoff = probe_workflow.prepare_probe_runtime_workspace(
    processed_dataset_root=PROCESSED_DATASET_ROOT,
    local_dataset_root=LOCAL_DATASET_ROOT,
    family_root=FAMILY_ROOT,
    run_root=RUN_ROOT,
    copy_processed_dataset=True,
)
PROCESSED_DATASET_MANIFEST = Path(runtime_workspace_handoff['local_dataset_manifest_path'])
if not PROCESSED_DATASET_MANIFEST.exists():
    raise FileNotFoundError(PROCESSED_DATASET_MANIFEST)
print(runtime_workspace_handoff)

## 04 Clone Or Update Repository

记录当前仓库 short commit, 设置 `PYTHONPATH`, 并打印 git 工作区状态, 便于结果包追踪到具体代码版本。

In [ ]:
repo_env = dict(os.environ)
repo_env['PYTHONPATH'] = str(REPO_ROOT)
GIT_COMMIT = subprocess.check_output(
    ['git', 'rev-parse', '--short', 'HEAD'],
    text=True,
    env=repo_env,
    cwd=REPO_ROOT,
).strip()
subprocess.run(['git', 'status', '--short'], check=True, env=repo_env, cwd=REPO_ROOT)

## 05 Install Runtime Dependencies

安装运行所需依赖, 包括测试工具、diffusers 相关依赖、模型加载依赖和视频质量指标依赖。

In [ ]:
subprocess.run(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        'pytest',
        'diffusers',
        'accelerate',
        'transformers',
        'safetensors',
        'huggingface_hub',
        'lpips',
        'pytorch-msssim',
    ],
    check=True,
    env=repo_env,
    cwd=REPO_ROOT,
)

## 06 Prepare Session Model

准备 session-only AutoencoderKL 模型目录, 并写出 session model manifest。

In [ ]:
session_model_manifest = probe_workflow.prepare_probe_session_model(
    model_id=MODEL_ID,
    local_model_root=LOCAL_MODEL_ROOT,
    session_manifest_path=SESSION_MODEL_MANIFEST_PATH,
    revision='main',
)
LOCAL_MODEL_ROOT = Path(session_model_manifest['models'][0]['local_path'])
if not LOCAL_MODEL_ROOT.exists():
    raise FileNotFoundError(LOCAL_MODEL_ROOT)
assert session_model_manifest['model_policy'] == 'session_only_no_drive_model_storage'

## 07 Write Runtime Config

输出 `RUNTIME_CONFIG_PATH` 是后续 runner、runtime manifest 与结果包审计的核心输入。


In [ ]:
runtime_config_handoff = probe_workflow.write_probe_runtime_config(
    runtime_config_path=RUNTIME_CONFIG_PATH,
    execution_environment='colab',
    processed_dataset_key=PROCESSED_DATASET_KEY,
    local_dataset_root=LOCAL_DATASET_ROOT,
    processed_dataset_root=PROCESSED_DATASET_ROOT,
    vae_model_local_path=LOCAL_MODEL_ROOT,
    dataset_manifest_path=PROCESSED_DATASET_MANIFEST,
    require_formal_pass_criteria=REQUIRE_FORMAL_PASS,
    extra_config={
        'family_id': FAMILY_ID,
        'workflow_key': WORKFLOW_KEY,
        'step_key': STEP_KEY,
        'git_commit': GIT_COMMIT,
        'runtime_manifest_overrides': {
            'family_root': str(FAMILY_ROOT),
            'session_model_manifest_path': str(SESSION_MODEL_MANIFEST_PATH),
        },
    },
)
print(runtime_config_handoff)

## 08 Check GPU And Runtime

作用：运行 runtime preflight, 检查当前运行模式、processed dataset 本地目录和 session model 目录是否满足运行前置条件。

In [ ]:
runtime_check_report = probe_workflow.run_probe_runtime_preflight(
    run_mode=RUN_MODE,
    local_dataset_root=LOCAL_DATASET_ROOT,
    local_model_dirs=[LOCAL_MODEL_ROOT],
)
print(runtime_check_report)

## 09 Verify Repository Contract

项目合同校验与总审计。


In [ ]:
subprocess.run([sys.executable, 'tools/harness/validate_project_contract.py'], check=True, env=repo_env, cwd=REPO_ROOT)
subprocess.run([sys.executable, 'tools/harness/run_all_audits.py'], check=True, env=repo_env, cwd=REPO_ROOT)

## 10 Run Smoke Tests

执行真实视频 VAE encode/decode smoke 测试, 判断视频读写、VAE metadata 与 digest 稳定性。

In [ ]:
subprocess.run(
    [
        sys.executable,
        '-m',
        'pytest',
        '-q',
        '-m',
        'smoke',
        'tests/integration/test_real_video_vae_encode_decode_smoke.py',
    ],
    check=True,
    env=repo_env,
    cwd=REPO_ROOT,
)

## 11 Run Real Video VAE Latent Formal

启动 probe runner, 由仓库模块生成 records、thresholds、manifests、artifacts 与后续可重建表格所需的证据。

`run_root=RUN_ROOT` 指定 session local 运行目录；`runtime_config_path=RUNTIME_CONFIG_PATH` 提供运行配置；`dataset_manifest=PROCESSED_DATASET_MANIFEST` 显式传入 processed dataset manifest, 防止回退到默认 manifest。

In [ ]:
probe_workflow.run_probe_runner(
    run_root=RUN_ROOT,
    run_mode=RUN_MODE,
    runtime_profile=RUNTIME_PROFILE,
    runtime_config_path=RUNTIME_CONFIG_PATH,
    dataset_manifest=PROCESSED_DATASET_MANIFEST,
    python_executable=sys.executable,
)

## 12 Rebuild Tables And Reports

records 与 manifests 重建 tables、reports、figures 和 failure gallery 等派生产物。

In [ ]:
probe_workflow.rebuild_probe_tables_and_reports(run_root=RUN_ROOT)

## 13 Check Real Video VAE Latent Outputs

校验输出完整性、formal pass 条件、runtime manifest、records-to-artifacts 链路与下一阶段门禁。

In [ ]:
formal_validation_summary = probe_workflow.check_probe_outputs(
    run_root=RUN_ROOT,
    construction_phase='real_video_vae_latent_probe',
    run_mode=RUN_MODE,
    require_formal_pass_criteria=REQUIRE_FORMAL_PASS,
)
if not formal_validation_summary['status']:
    raise RuntimeError(formal_validation_summary)

## 14 Package Family Results

生成 family 结果包, 并可选写入 Drive 侧 result registry 与 family registry。

`family_root=FAMILY_ROOT` 决定结果包归档位置；
`formal_validation_summary` 是打包前置证据；
`family_id`、`workflow_key`、`step_key` 用于登记本次正式运行身份。

In [ ]:
package_payload = probe_workflow.package_probe_family_results(
    run_root=RUN_ROOT,
    family_root=FAMILY_ROOT,
    require_formal_pass_criteria=REQUIRE_FORMAL_PASS,
    formal_validation_summary=formal_validation_summary,
    drive_root=DRIVE_ROOT,
    family_id=FAMILY_ID,
    workflow_key=WORKFLOW_KEY,
    step_key=STEP_KEY,
    run_mode=RUN_MODE,
)
drive_archive_path = package_payload['drive_archive_path']
compat_pack_root = package_payload['compat_pack_root']
print({
    'drive_archive_path': drive_archive_path,
    'compat_pack_root': compat_pack_root,
    'result_registry.jsonl': package_payload.get('registry_paths', {}).get('result_registry.jsonl'),
    'family_registry.jsonl': package_payload.get('registry_paths', {}).get('family_registry.jsonl'),
})

## 15 Print Final Family Summary

打印本次摘要。

In [ ]:
print({
    'family_id': FAMILY_ID,
    'run_root': str(RUN_ROOT),
    'family_root': str(FAMILY_ROOT),
    'formal_validation_summary': formal_validation_summary,
    'drive_archive_path': str(drive_archive_path),
    'result_registry.jsonl': package_payload.get('registry_paths', {}).get('result_registry.jsonl'),
    'family_registry.jsonl': package_payload.get('registry_paths', {}).get('family_registry.jsonl'),
})